In [1]:
import os.path as osp

from utils import ConstantWarmupScheduler, save_figure_loss, save_figure_auc
import os
import torchvision.transforms as tt
from ADA import task, train
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from models import CoOp
import torch, argparse, os
import torch.nn as nn

import random
import numpy as np
import wandb

from dataloader import model_aware_load
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
import pandas as pd
import math

In [2]:
def find_all_pt_files(directory="result"):

    pt_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.pt'):
                pt_files.append(os.path.join(root, file))
    return pt_files

In [3]:
data = [ os.path.join("../../data/val", path) for path in os.listdir("../../data/val") if "png" in path]
data[0]

'../../data/val/8997.png'

In [19]:
data = sorted(data, key=lambda x: int(x.split('/')[-1].split('.')[0]))
data[:5]

['../../data/val/0.png',
 '../../data/val/1.png',
 '../../data/val/2.png',
 '../../data/val/3.png',
 '../../data/val/4.png']

In [20]:
class OPT:
    model = "CoOp"
    backbone = "ViT"
    n_ctx = 32
    train = "store_true"
    train_real = "coco"
    n_classes = 2
    pos_name = "fake"
    neg_name = "real"
    #### set up !!
    train_target = None

In [21]:
class MyDataset(Dataset): 
    
    def __init__(self, data, transform):
        print("init dataset for : ", len(data))
        self.data = [Image.open(img).convert("RGB") for img in tqdm(data)]
        self.transform_norm = transform
    
    def __getitem__(self, idx):
        image = self.data[idx]   # Image.open( self.data[idx]).convert("RGB") 
        image = self.transform_norm(image)

        return image
    
    def __len__(self) -> int:
        return len(self.data)

In [22]:
opt = OPT()
paths = find_all_pt_files()
class2path = {
    str(p[:-3].split("_")[-1]): p 
    for p in paths
}

In [23]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

all_preds = []
targets = ['sd21', 'sdxl', 'sd3', 'dalle', 'midjourney']
for target in targets:

    opt.train_target = target

    model, transform = CoOp(opt)

    state_dict = torch.load(class2path[target], map_location=torch.device('cpu'))
    model.load_state_dict(state_dict)

    model = model.to(device)     
    valid_dataset = MyDataset(data, transform)
    valid_loader = DataLoader(valid_dataset, batch_size=1024, shuffle=False)

    model.eval()
    preds = []

    with torch.no_grad():
        for inputs in tqdm(valid_loader):
            inputs = inputs.to(device)

            features_logits, _, _ = model(inputs)
            prompts_loss = torch.zeros(1).to(features_logits.device)

            if len(features_logits.shape) == 2:
                temp_features_logits = features_logits
                preds += torch.nn.functional.softmax(temp_features_logits, dim=1).tolist()
            elif len(features_logits.shape) == 3:
                temp_features_logits = features_logits[:, 0:1, :].squeeze(1)
                preds += torch.nn.functional.softmax(temp_features_logits, dim=1).tolist()
    
    preds = np.array(preds)
    preds = preds[:, 1].reshape(-1)
    all_preds.append(preds)

all_preds = np.array(all_preds)
all_preds = all_preds.T
all_preds.shape

/tmp/ipykernel_1206565/2071424166.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(class2path[target], map_location=torch.device('cpu'))


Building custom CLIP
Initializing a generic context
Initial context: "X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X"
Number of context words (tokens): 32
Prompts are: ['X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X real.', 'X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X fake.']
init dataset for :  9000


100%|██████████| 9/9 [01:02<00:00,  6.98s/it]


Building custom CLIP
Initializing a generic context
Initial context: "X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X"
Number of context words (tokens): 32
Prompts are: ['X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X real.', 'X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X fake.']
init dataset for :  9000


100%|██████████| 9/9 [01:01<00:00,  6.88s/it]


...

Building custom CLIP
Initializing a generic context
Initial context: "X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X"
Number of context words (tokens): 32
Prompts are: ['X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X real.', 'X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X fake.']
init dataset for :  9000


100%|██████████| 9/9 [01:02<00:00,  6.89s/it]


Building custom CLIP
Initializing a generic context
Initial context: "X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X"
Number of context words (tokens): 32
Prompts are: ['X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X real.', 'X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X X fake.']
init dataset for :  9000


100%|██████████| 9/9 [01:03<00:00,  7.03s/it]


(9000, 5)

In [24]:
all_preds

array([[9.95724201e-01, 9.99782741e-01, 9.99862909e-01, 8.64914358e-02,
        9.93523240e-01],
       [6.75342619e-07, 3.27030349e-08, 3.39828254e-08, 9.32721955e-11,
        1.00821582e-08],
       [9.98152316e-01, 4.20470387e-01, 9.98779118e-01, 4.19971980e-02,
        9.23377350e-02],
       ...,
       [9.24461460e-07, 8.78903990e-08, 1.23590307e-05, 7.60857120e-07,
        1.05500305e-06],
       [9.94532704e-01, 8.83554459e-01, 9.98370349e-01, 1.00000000e+00,
        9.98459697e-01],
       [9.94630933e-01, 9.99949217e-01, 9.99928713e-01, 2.99747899e-05,
        1.24826498e-01]])

In [25]:
max_category = np.argmax(all_preds, axis=1) + 1
max_values = np.max(all_preds, axis=1)


In [26]:
multi_preds = []

for pred, value in zip(max_category, max_values):
    if value > 0.5:
        multi_preds.append(pred)
    else:
        multi_preds.append(0)

len(multi_preds)

9000

In [27]:
val_shuffle_df = pd.read_csv(os.path.join("..", "..", "data", "val", "val_shuffle.csv"))
val_shuffle_df.head()

,Caption,Image,Label_A,Label_B
0,a toilet sits next to a shower an sink,/home/nasrin/HPCServer/AIISC/AGID/dataset/COCO...,1,3
1,A TV sitting on top of a table next to a lapto...,/home/nasrin/HPCServer/AIISC/AGID/dataset/COCO...,0,0
2,Two giraffes eat from a pot attached to a fence.,/home/nasrin/HPCServer/AIISC/AGID/dataset/COCO...,1,1
3,The kitchen has many grill with pots hanging a...,/home/nasrin/HPCServer/AIISC/AGID/dataset/COCO...,1,4
4,some road signs besides a road in the street,/home/nasrin/HPCServer/AIISC/AGID/dataset/COCO...,1,1


In [28]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

binary_preds = [ 0 if label == 0 else 1 for label in multi_preds]
print("binary accuracy: ", accuracy_score(val_shuffle_df["Label_A"], binary_preds))
print("binary f1 score: ", f1_score(val_shuffle_df["Label_A"], binary_preds, average="macro"))

binary accuracy:  0.9934444444444445
binary f1 score:  0.9881334309207794


In [31]:
print("multi accuracy: ", accuracy_score(val_shuffle_df["Label_B"], multi_preds))
print("multi f1 score: ", f1_score(val_shuffle_df["Label_B"], multi_preds, average="macro"))

multi accuracy:  0.8693333333333333
multi f1 score:  0.8721154459486177
